- Возможно можно использовать просмотры как признаки. Но у каждого должен быть свой вес, если так можно.
- Для фильтрации: по айтему можно посмотреть, покупают ли его повторно. Если да, то дать признак повторный.
- Возможно не давать рекомендации для айтемов, которые available == 0
- Один айтем может иметь несколько записей о категориях. Логичнее использовать последнюю.
- Можно использовать подход из курса: коллаб als, контент, признаки и catboost.
- Нужно создать отдельный df с категориями

- надо засунуть графики, куда можно

# EDA
- Items:
1. В таблице item_properties можно проследить, что у 6% товаров есть несколько записей о категориях, вероятнее всего они принадлежат нескольким категориям сразу.
2. У категорий айтемов достаточно сложная система. В отдельной таблице можно пройти от подкатегории товара до общей категории. Если использовать контентную матрицу на основе категорий, то необходимо выбрать подход: подкатегории или общие. При выборе подкатегорий рекомендации будут более точными, но некоторые товары с похожими категориями могут не быть отмечены. При выборе изначальных будет больше связей между товарыми, но рекомендации будут менее точными с связи с отсутствием подкатегорий.  
Поскольку для системы рекомендаций будет использоваться двухстадийный подход с признаками, имеет смысл выбрать подкатегории.
- Events:
1. Начало данных: 2015-05-03, конец: 2015-09-18
1. В датасете есть очень большой дисбаланс типов интеракций: 97% просмотров, 2% добавлений в корзину, ~1% покупок. В связи с этим, построение als матрицы только на основе таргета невозможно. 
2. В датасете присутствует слабая корреляция кол-ва просмотров с добавлениями в корзину. На основе этого можно использовать просмотр как слабый вес (0.3) при построении матрицы als с добавлением в корзину (1.0).
# Обработка данных:
- объединение item_properties в один df
- замена timestamp на формат даты
- создание отдельного df с айтемами и их категориями
- добавление колонки target в соответствии с ивентом addtocart

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## item_properties

In [9]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# трансформация даты
item_properties_part1 = pd.read_csv('item_properties_part1.csv')
item_properties_part1['date'] = pd.to_datetime(item_properties_part1['timestamp'], unit='ms')
item_properties_part2 = pd.read_csv('item_properties_part2.csv')
item_properties_part2['date'] = pd.to_datetime(item_properties_part2['timestamp'], unit='ms')

# объединения датафреймов
item_properties = pd.concat([item_properties_part1, item_properties_part2]).drop(columns=['timestamp'])

In [ ]:
# айтемы с несколькими категориями

df_cat = item_properties[item_properties['property'] == 'categoryid']
df_cat = df_cat.groupby('itemid').count()
df_cat = df_cat[df_cat['property'] > 1]
df_cat.shape

(23352, 3)

In [ ]:
# создание отдельного датафрейма с категориями

df_cat = item_properties[item_properties['property'] == 'categoryid'].sort_values('date').drop(columns=['property'])
df_cat = df_cat.groupby('itemid')['value'].last().reset_index(name='category')

In [ ]:
# самые популярные категории

genre_count = df_cat.value_counts('category').reset_index(name='count')
genre_count.head(10)

,category,count
0,342,17231
1,769,10982
2,173,10561
3,1301,9943
4,1007,9737
5,1142,6818
6,1680,6019
7,1250,5193
8,1070,4209
9,1483,3868


In [ ]:
item_properties['itemid'].nunique()

417053

## category_tree

In [ ]:
df3 = pd.read_csv('category_tree.csv')
df3.sample(10)

## events

In [13]:
import pandas as pd

events = pd.read_csv('events.csv')
events['date'] = pd.to_datetime(events['timestamp'], unit='ms')
events = events.drop(columns=['timestamp'])

In [ ]:
# распределение типов интеракций

events['event'].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [ ]:
events.shape

(2756101, 5)

In [ ]:
# кол-во юзеров с добавлением в корзину

un = events[events['event'] == 'addtocart']
un['visitorid'].nunique()

37722

In [ ]:
# кол-во айтемов с добавлением в корзину

un = events[events['event'] == 'addtocart']
un['itemid'].nunique()

23903

In [ ]:
# общее кол-во юзеров

events['visitorid'].nunique()

1407580

In [17]:
#корелляция просмотров с таргетом

viewcount = events[events['event'] == 'view']
viewcount = viewcount.groupby('visitorid')['itemid'].value_counts().reset_index(name='viewcount')

cartcount = events[events['event'] == 'addtocart']
cartcount = cartcount.groupby('visitorid')['itemid'].value_counts().reset_index(name='cartcount')

oor = pd.merge(viewcount, cartcount[['visitorid', 'itemid', 'cartcount']], how='left', on=['visitorid', 'itemid'])

In [5]:
oor[['viewcount', 'cartcount']] = oor[['viewcount', 'cartcount']].fillna(0)
correlation = oor['viewcount'].corr(oor['cartcount'], method='spearman')
print(correlation)

0.17998033406404654
